# PALS Phase 2: Causal Circuit Discovery (OLMo-3.1 32B fp16)

Causal experiments H18-H21 to address the limitations of correlational analysis.
Builds on the 17-experiment results from Phase 1.

In [ ]:
import subprocess, sys, os

if not os.path.exists('/content/clin-syco'):
    subprocess.run(['git', 'clone', 'https://github.com/elliott-leow/clin-syco.git', '/content/clin-syco'], check=True)

os.chdir('/content/clin-syco')
subprocess.run(['git', 'pull', '--ff-only'], check=False)

import importlib, sys as _sys
for key in list(_sys.modules.keys()):
    if key.startswith('experiments.') or key.startswith('pals.'):
        del _sys.modules[key]

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)

import torch
print(f'PyTorch {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
import gc, json, os, time, platform, sys, torch
from datetime import datetime, timezone

sys.path.insert(0, '/content/clin-syco/src')
sys.path.insert(0, '/content/clin-syco')

DEVICE = 'cuda'
STIMULI_DIR = '/content/clin-syco/stimuli'
OUTPUT_DIR = '/content/clin-syco/results/olmo3_32b_fp16_causal'
os.makedirs(OUTPUT_DIR, exist_ok=True)

DPO_MODEL = 'allenai/Olmo-3.1-32B-Instruct-DPO'
ANALYSIS_LAYERS = list(range(0, 64, 4)) + [63]

def cleanup():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def vram(): return torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f'Loading {DPO_MODEL} (fp16)...')
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(DPO_MODEL, torch_dtype=torch.float16,
                                              device_map='auto', attn_implementation='sdpa')
model.eval()
tokenizer = AutoTokenizer.from_pretrained(DPO_MODEL)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print(f'Loaded in {time.time()-t0:.0f}s, VRAM: {vram():.1f} GB')

In [ ]:
# H18: Causal Tracing
from experiments.h18_causal_tracing import run as run_h18

t0 = time.time()
h18_dir = os.path.join(OUTPUT_DIR, 'h18')
run_h18(model, tokenizer, STIMULI_DIR, h18_dir, layers=ANALYSIS_LAYERS, n_stimuli=10, n_completion_tokens=3)
cleanup()
print(f'H18 done in {time.time()-t0:.0f}s')

In [ ]:
# Load H18 results to get critical layers
import json
with open(os.path.join(OUTPUT_DIR, 'h18/h18_results.json')) as f:
    h18 = json.load(f)
critical = h18['critical_layers']
print(f'Critical layers from H18: {critical}')

# H19: Path Patching on critical layers only
from experiments.h19_path_patching import run as run_h19

t0 = time.time()
h19_dir = os.path.join(OUTPUT_DIR, 'h19')
run_h19(model, tokenizer, STIMULI_DIR, h19_dir, layers=ANALYSIS_LAYERS, n_stimuli=5, critical_layers=critical)
cleanup()
print(f'H19 done in {time.time()-t0:.0f}s')

In [ ]:
# H20: Circuit-Based Steering
from experiments.h20_circuit_steering import run as run_h20

t0 = time.time()
h20_dir = os.path.join(OUTPUT_DIR, 'h20')
run_h20(model, tokenizer, STIMULI_DIR, h20_dir, layers=ANALYSIS_LAYERS, n_stimuli=10,
        critical_layers=critical, alphas=[2.0, 4.0, 8.0])
cleanup()
print(f'H20 done in {time.time()-t0:.0f}s')

In [ ]:
# H21: Nonlinear Steering
from experiments.h21_nonlinear_steering import run as run_h21

t0 = time.time()
h21_dir = os.path.join(OUTPUT_DIR, 'h21')
run_h21(model, tokenizer, STIMULI_DIR, h21_dir, layers=ANALYSIS_LAYERS, n_stimuli=15,
        hidden_size=32, epochs=30)
cleanup()
print(f'H21 done in {time.time()-t0:.0f}s')

In [ ]:
# Download results
import shutil
zip_path = '/content/pals_causal_results'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f'Archive: {zip_path}.zip ({os.path.getsize(zip_path + ".zip") / 1e6:.1f} MB)')
try:
    from google.colab import files
    files.download(zip_path + '.zip')
except ImportError:
    print(f'Download: {zip_path}.zip')